In [1]:
import os
import sys
# Agregar el directorio raíz del proyecto al path
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..','..'))
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)
    
    
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *   
from services.utilidades import *
from services.comprobar_netcdf import *

In [2]:
# Recargar módulos locales (ejecutar esta celda cuando cambies código en services/)
import importlib
import sys

# Lista de módulos locales a recargar
modulos_locales = [
    'services.carga_de_datos',
    'services.DataFrame_maker',
    'services.prueba',
    'services.utilidades',
    'services.get_tspan',
    'services.polar2uv',
    'services.uv2polar'
]

for modulo in modulos_locales:
    if modulo in sys.modules:
        importlib.reload(sys.modules[modulo])
        print(f"✓ Recargado: {modulo}")
    else:
        print(f"- No cargado: {modulo}")

# Re-importar después de recargar
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *

print("\n¡Módulos locales recargados!")

✓ Recargado: services.carga_de_datos
✓ Recargado: services.DataFrame_maker
- No cargado: services.prueba
✓ Recargado: services.utilidades
✓ Recargado: services.get_tspan
✓ Recargado: services.polar2uv
✓ Recargado: services.uv2polar

¡Módulos locales recargados!


In [3]:
# %matplotlib tk

In [4]:
dict_de_boyas = {}

### 2. OLEAJE

In [ ]:
p1 = pd.date_range('2026-03-07 00:00', '2026-06-27 07:00', freq='h') 
print(f"p1 = {len(p1)}")

p1 = 2849


In [6]:
# p1: Crudo unido pkl
nombre = "BMT3-03-T80_pkl"
archivos_csv = "msj3_crudo_unido_realt_2026030700_2026062707.pkl"
ruta_a_carpeta = "\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_4.2\\2026\\07. Julio\\BMT3-03-T80\\DATOS"
df_pkl_crudo_realt = cargar_pickle_oleaje(ruta_a_carpeta, archivos_csv)
dict_de_boyas[nombre] = df_pkl_crudo_realt # Guardo el dataframe dentro del diccionario de boyas.
df_pkl_crudo_realt.head(5)

,tspan,Hs,Hmax,Tp,dir,vardir
0,2026-03-07 00:00:00,0.59,1.08,4.5,59.7,41.2
1,2026-03-07 01:00:00,0.64,1.04,5.0,51.0,40.8
2,2026-03-07 02:00:00,NaN,NaN,NaN,NaN,NaN
3,2026-03-07 03:00:00,0.70,1.19,4.3,59.5,45.0
4,2026-03-07 04:00:00,0.70,1.11,4.0,63.7,41.7


In [7]:
df_pkl_crudo_realt.tail(2)

,tspan,Hs,Hmax,Tp,dir,vardir
2694,2026-06-27 06:00:00,0.92,1.37,5.4,59.2,30.1
2695,2026-06-27 07:00:00,0.92,1.67,5.6,57.8,30.5


In [8]:
len(df_pkl_crudo_realt)

2696

In [9]:
# 1.5. p2: Cargar NETCDF 
nombre = "BMT3-03-T80_NC"
nombre_de_archivo = "BMT3-03-T80-TRIAXYSG3-NS09282-Z0-REALT-20260307-20260627.nc"
df_nc, nc, DirSpec = cargar_nc_oleaje(ruta_a_carpeta, nombre_de_archivo)
dict_de_boyas[nombre] = df_nc # Guardo el dataframe dentro del diccionario de boyas.
df_nc.head(5)

,tspan,Hs,Hmax,Tp,dir,vardir
0,2026-03-07 00:00:00,0.59,1.08,4.5,59.7,41.2
1,2026-03-07 01:00:00,0.64,1.04,5.0,51.0,40.8
2,2026-03-07 02:00:00,NaN,NaN,NaN,NaN,NaN
3,2026-03-07 03:00:00,0.70,1.19,4.3,59.5,45.0
4,2026-03-07 04:00:00,0.70,1.11,4.0,63.7,41.7


In [10]:
df_nc.tail(2)

,tspan,Hs,Hmax,Tp,dir,vardir
2694,2026-06-27 06:00:00,0.92,1.37,5.4,59.2,30.1
2695,2026-06-27 07:00:00,0.92,1.67,5.6,57.8,30.5


In [11]:
# Otras revisiones del NC:
# comprobar_nombre_nc
comprobar_nombre_nc(nombre_de_archivo, nc)
comprobar_porcentaje_de_variable(nc, ruta_a_carpeta, "BMT3-03-T80_datos_word_2026030700_2026062707.xlsx", variables="oleaje")

NAM correcto


,Variable,esperados,validados,porcentaje_validados
0,Hs,2696,2692.0,99.851632
1,Hm,2696,2692.0,99.851632
2,Tp,2696,2690.0,99.777448
3,dir,2696,2693.0,99.888724
4,vardir,2696,2693.0,99.888724
5,dirspec,2696,2634.0,97.700297


In [12]:
# 1.5. p2: Cargar telemetría de BOT1-03-T50 
nombre = "BOT1-03-T50"
df_pkl = pd.DataFrame()  # Inicializar un DataFrame vacío para almacenar los datos combinados
for mes in ["01. Enero","02. Febrero","03. Marzo","04. Abril","05. Mayo", "06. Junio"]:
    ruta_a_carpeta = f"\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_1.4\\2026\\{mes}\\BOT1-03-T50\\DATOS"
    archivos = os.listdir(ruta_a_carpeta)
    for archivo in archivos:
        if archivo.endswith(".pkl"):
            if archivo.startswith("msj3_crudo_unido_realt"):
                df_pkl_crudo_realt = cargar_pickle_oleaje(ruta_a_carpeta, archivo)
                df_pkl = pd.concat([df_pkl, df_pkl_crudo_realt], ignore_index=True)
    
df_pkl = df_pkl.reset_index(drop=True)  # Reiniciar el índice del DataFrame combinado
dict_de_boyas[nombre] = df_pkl # Guardo el dataframe combinado dentro del diccionario de boyas.

In [13]:
# 1.5. p2: Cargar telemetría de BOT1-04-T80 
nombre = "BOT1-04-T80"
df_pkl = pd.DataFrame()  # Inicializar un DataFrame vacío para almacenar los datos combinados
for mes in ["01. Enero","02. Febrero","03. Marzo","04. Abril","05. Mayo", "06. Junio"]:
    ruta_a_carpeta = f"\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_1.4\\2026\\{mes}\\BOT1-04-T80\\DATOS"
    archivos = os.listdir(ruta_a_carpeta)
    for archivo in archivos:
        if archivo.endswith(".pkl"):
            if archivo.startswith("msj3_crudo_unido_realt"):
                df_pkl_crudo_realt = cargar_pickle_oleaje(ruta_a_carpeta, archivo)
                df_pkl = pd.concat([df_pkl, df_pkl_crudo_realt], ignore_index=True)
    
df_pkl = df_pkl.reset_index(drop=True)  # Reiniciar el índice del DataFrame combinado
dict_de_boyas[nombre] = df_pkl # Guardo el dataframe combinado dentro del diccionario de boyas.

In [14]:
# 1.5. p2: Cargar telemetría de BOT1-05-T50
nombre = "BOT1-05-T50"
df_pkl = pd.DataFrame()  # Inicializar un DataFrame vacío para almacenar los datos combinados
for mes in ["01. Enero","02. Febrero","03. Marzo","04. Abril","05. Mayo", "06. Junio"]:
    ruta_a_carpeta = f"\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_1.4\\2026\\{mes}\\BOT1-05-T50\\DATOS"
    archivos = os.listdir(ruta_a_carpeta)
    for archivo in archivos:
        if archivo.endswith(".pkl"):
            if archivo.startswith("msj3_crudo_unido_realt"):
                df_pkl_crudo_realt = cargar_pickle_oleaje(ruta_a_carpeta, archivo)
                df_pkl = pd.concat([df_pkl, df_pkl_crudo_realt], ignore_index=True)
    
df_pkl = df_pkl.reset_index(drop=True)  # Reiniciar el índice del DataFrame combinado
dict_de_boyas[nombre] = df_pkl # Guardo el dataframe combinado dentro del diccionario de boyas.

In [15]:
# Crear el diccionario de boyas con los dataframes cargados
df = crear_dataframe_de_boyas(dict_de_boyas)

Index(['tspan', 'Hs_BMT3-03-T80_pkl', 'Hmax_BMT3-03-T80_pkl',
       'Tp_BMT3-03-T80_pkl', 'dir_BMT3-03-T80_pkl', 'vardir_BMT3-03-T80_pkl',
       'Hs_BMT3-03-T80_NC', 'Hmax_BMT3-03-T80_NC', 'Tp_BMT3-03-T80_NC',
       'dir_BMT3-03-T80_NC', 'vardir_BMT3-03-T80_NC', 'Hs_BOT1-03-T50',
       'Hmax_BOT1-03-T50', 'Tp_BOT1-03-T50', 'dir_BOT1-03-T50',
       'vardir_BOT1-03-T50', 'Hs_BOT1-04-T80', 'Hmax_BOT1-04-T80',
       'Tp_BOT1-04-T80', 'dir_BOT1-04-T80', 'vardir_BOT1-04-T80',
       'Hs_BOT1-05-T50', 'Hmax_BOT1-05-T50', 'Tp_BOT1-05-T50',
       'dir_BOT1-05-T50', 'vardir_BOT1-05-T50'],
      dtype='str')


In [16]:
app = InteractivePlotterTk(df)
# Iniciar el bucle de la aplicación
app.root.mainloop()

# NOTAS PENDIENTES:
# - Arreglar que al eliminar puntos para que no tenga límite el historial de deshacer
# - Arreglar para que solo se grafiquen las columnas elegidas por el usuario
# - Agregar una función que grafique las variables de Rap y Dir --> sería algo como que df.iloc[idx,col][prof]
# - Arreglar el caso de cuando el usuario elimina puntos de Rap o Dir colocar U y V como nans --> df.iloc[idx,col][prof] = np.nan
# - Agregar la funcionalidad de guardar el DataFrame corregido en un archivo .csv o .xlsx, pkl
# - Agregar la funcionalidad de devolver puntos eliminados (no deshacer la última acción, sino devolver puntos específicos)

In [17]:
# idxs = app.deletion_log["vardir_BOT1-04-T80_pkl_realt_V"]["idxs"]
# tspan_num = mdates.num2date(app.data.loc[idxs,"tspan"])
# tspan_pd = pd.to_datetime(tspan_num)
# df_out = pd.DataFrame({"tspan": tspan_pd})
# df_out.to_csv("tspan_eliminados.csv", index=False)

